# Protocol 2 — the retrieval half, step by step

The engines of protocol 2 do not search abstracts: they search **passages** (windows of
consecutive sentences, 100 words at most, title in front), and the model will later read
exactly the passage that was scored. `method.py` builds all of it; this notebook imports
its functions and follows one claim through the chain, **claim 1140**.

Run `python method.py` once before this notebook. On the first run it downloads the
original SciFact release into `data/` (3 MB, only the three files we read) — the testbed
corpus only keeps a joined text field, and we need the sentence-split abstracts and the
annotated evidence. It then writes `bm25_run.json`, `bge_run.json`, their
`*_best_passage.json` companions, `draws.json` and `dry_run.txt`, asserts the frozen
control numbers, and never overwrites: a recomputation must give back the same content
(the BGE run, not bit-reproducible across machines, is encoded once — about 8 minutes on
CPU — then read back).

In [1]:
import json
import method
import testbed

method.download_data()
doc_ids, titles, sentences = method.load_corpus()
queries, qrels = method.load_queries()
claims = method.load_claims("dev")     # the BEIR test split is the original dev
sisters = method.sister_evidence(claims)
print(len(doc_ids), "abstracts,", len(queries), "test queries,", len(claims), "labelled claims")

5183 abstracts, 300 test queries, 188 labelled claims


## The claim we follow

A CONTRADICT claim with a single gold abstract. The gold is an abstract with annotated
evidence, not just a cited abstract: 13 test claims also cite abstracts that the
annotators judged evidence-free, and those are *not* golds.

In [2]:
claim = claims["1140"]
print(claim["text"])
print("label:", claim["label"], " golds:", claim["golds"], " cited:", sorted(claim["cited"]))

Taking 400mg of α-tocopheryl acetate helps to prevent prostate cancer.
label: CONTRADICT  golds: ['12009265']  cited: ['12009265']


## From abstracts to passages

An abstract becomes all its **maximal** windows of consecutive sentences totalling at most
100 words. Windows overlap. A sentence longer than 100 words fits in no window (26 such
sentences are dropped) and the corpus's one single-word passage ("...") is dropped too,
which leaves **24 917 passages** for 5 183 abstracts.

Our example: abstract 41710132 has 6 sentences and gives 3 windows. A passage id is
`doc:a-b` with `b` excluded, so `41710132:2-6` shows sentences 2 to 5.

In [3]:
passages, windows = method.build_passages(doc_ids, sentences)
print(len(passages), "passages")
pos = {d: j for j, d in enumerate(doc_ids)}
j = pos["41710132"]
print("sentence lengths of 41710132:", [len(s.split()) for s in sentences["41710132"]], "words")
print("its windows:", windows[j])

24917 passages
sentence lengths of 41710132: [24, 48, 30, 22, 10, 27] words
its windows: [(0, 2), (1, 4), (2, 6)]


## BM25 on passages, one score per abstract (MaxP)

The index uses the testbed's BM25: k1=1.2, b=0.75, and the testbed's own preprocessing
line (sklearn analyzer, NLTK stopwords, Porter stems). Every passage gets a score and an
abstract is represented by its best passage. For claim 1140 the three windows of 41710132
score very differently — the abstract enters the ranking with the last one:

In [4]:
bm25 = method.build_index(passages, doc_ids, titles, sentences)
scores = bm25.get_scores(method.preprocess(queries["1140"]))
for a, b in windows[j]:
    p = passages.index((j, a, b))
    print(f"passage {a}-{b}: score {scores[p]:.4f}")

passage 0-2: score 0.0000
passage 1-4: score 4.7968
passage 2-6: score 13.7387


## The run

Abstracts ranked by their best passage, whatever engine produced the scores. Ties are
broken by corpus order, and inside an abstract by the leftmost window, so the files come
out identical on any machine. For claim 1140 the gold sits at rank 1 and 41710132 right
behind it — it will be distractor n°1. The run file keeps the 101 best abstracts of each
of the 300 test queries (`{claim: {abstract: score}}`), so the testbed can score it like
any run; the companion file remembers which passage produced each score.

In [5]:
order, best_score, best_window = method.rank_abstracts(scores, passages)
for rank in range(5):
    x = order[rank]
    a, b = best_window[x]
    tag = "gold" if doc_ids[x] in claim["golds"] else "    "
    print(rank + 1, tag, doc_ids[x], f"score {best_score[x]:.2f}", f" best passage {a}-{b}")

1 gold 12009265 score 19.95  best passage 0-4
2      41710132 score 13.74  best passage 2-6
3      26067999 score 12.82  best passage 66-71
4      24581365 score 12.51  best passage 9-12
5      18473550 score 11.93  best passage 3-7


## Does searching in passages hurt the ranking?

The run file is scored by the course's shared tool (`testbed.score_run`, pytrec_eval
against the BEIR qrels of the 300 test queries), and compared with the testbed's own
BM25 (title + first 200 words) by the shared paired test. The difference is not
significant: searching in passages does not degrade the abstract ranking.

In [6]:
with open(method.RUN_FILE) as f:
    run = json.load(f)
ndcg, recall = method.evaluate(run)
print(f"nDCG@10 {ndcg:.3f}  Recall@100 {recall:.3f}")
qids, nd, rc = testbed.score_run(run, "scifact")
kept = testbed.stored()["scifact"]      # the per-query baseline scores shipped with the testbed
assert kept["qids"] == qids
print("vs the testbed BM25 (nDCG@10):", testbed.compare(nd, kept["bm25"]["ndcg10"]))

nDCG@10 0.691  Recall@100 0.921
vs the testbed BM25 (nDCG@10): {'delta': 0.015, 'p': 0.07282807158801065, 'ci95': [-0.0011, 0.0316]}


## Distractors

Read the run: drop the golds first, then take the 18 best remaining abstracts, each shown
by its best passage. (Never "take 18 then drop golds": for most claims a gold sits at the
very top of the ranking — counted below — so that order would leave claims short.)
Abstracts that are annotated evidence of a *sister* claim are skipped like golds — next
section. The draws land in `draws.json`, recomputed here from the two run files to show
nothing else happens in between:

In [7]:
with open(method.BEST_FILE) as f:
    best_pass = json.load(f)
with open(method.DRAWS_FILE) as f:
    draws = json.load(f)
print(draws["1140"]["bm25"][:6])
live = method.pick_distractors(run["1140"], best_pass["1140"], set(claim["golds"]), sisters["1140"])
print("file = recomputation from the run files:", live == draws["1140"]["bm25"])
lead = 0
for qid in claims:
    top = list(run[qid])[0]
    lead += top in claims[qid]["golds"]
print("a gold is the top-ranked abstract for", lead, "of the 188 claims")
doc, a, b = method.split_id(draws["1140"]["bm25"][0])
print()
print("distractor n°1, as the model will read it:")
print(method.passage_text(titles, sentences, doc, a, b))

['41710132:2-6', '26067999:66-71', '24581365:9-12', '18473550:3-7', '25915873:0-4', '29981186:13-16']
file = recomputation from the run files: True
a gold is the top-ranked abstract for 136 of the 188 claims

distractor n°1, as the model will read it:
Regulation of E2Fs and senescence by PML nuclear bodies. Blocking the functions of the Rb protein family or adding back E2Fs to PML-expressing cells can rescue their defects in E2F-dependent gene expression and cell proliferation, inhibiting the senescent phenotype. In benign prostatic hyperplasia, a neoplastic disease that displays features of senescence, PML was found to be up-regulated and forming nuclear bodies. In contrast, PML bodies were rarely visualized in prostate cancers. The newly defined PML/Rb/E2F pathway may help to distinguish benign tumors from cancers, and suggest E2F target genes as potential targets to induce senescence in human tumors.


## Why the sister filter

SciFact contains paraphrases and negations of a same claim, and they do not always cite
the same abstracts. Claim 781 (test) says that mice lacking interferon-γ strongly resist
autoimmune myocarditis; claim 780 (original train split) is its negation and cites
abstract 8246922 as evidence. For 781, BM25 ranks that abstract first: a "distractor"
that actually settles the claim. So the draws skip every abstract that is annotated
evidence of a sister claim (a claim of the original train/dev citing at least one same
abstract). Cost over the 188 claims: 7 distractors replaced, in 5 claims.

In [8]:
bare = method.pick_distractors(run["781"], best_pass["781"], set(claims["781"]["golds"]), set())
print("without the filter:", bare[0], " <- annotated evidence of sister claim 780")
print("with the filter:   ", draws["781"]["bm25"][0])
for qid in ("540", "598", "781", "971"):
    docs = set()
    for pid in draws[qid]["bm25"]:
        docs.add(method.split_id(pid)[0])
    assert not docs & sisters[qid]
print("claims 540/598/781/971: no sister evidence left among their distractors")

without the filter: 8246922:2-8  <- annotated evidence of sister claim 780
with the filter:    40632104:1-6
claims 540/598/781/971: no sister evidence left among their distractors


## Randoms

Regime B, one shared `random.Random(10)`, claims in ascending id order: draw an abstract
with probability proportional to its number of disjoint 100-word blocks — the chance a
uniform passage draw gives it in the paper's corpus, where passages are disjoint — then
one of its windows uniformly, right away (the whole sequence is frozen by the seed).
Abstracts are distinct within a draw; cited abstracts and sister evidence are excluded;
**no content filter** beyond that, faithful to the paper. 18 per claim, precomputed once
and shared by all models and conditions.

In [9]:
print(claim["text"])
print()
for pid in draws["1140"]["random"]:
    print(pid)
doc, a, b = method.split_id(draws["1140"]["random"][0])
print()
print("the first one, as the model will read it:")
print(method.passage_text(titles, sentences, doc, a, b))

Taking 400mg of α-tocopheryl acetate helps to prevent prostate cancer.

10704438:5-9
7711685:1-5
31851367:1-4
1102268:3-7
11271123:7-12
275294:1-5
8318922:3-8
87337034:2-11
6793674:2-8
1538080:3-7
799586:5-8
7820043:0-5
243694:2-4
21943141:1-4
4421746:6-10
198309074:7-11
39424916:3-7
9648896:1-6

the first one, as the model will read it:
The DNA damage response during mitosis. Early on it was recognized that cells which are irradiated during mitosis continued division without repairing broken chromosomes. Although these initial observations indicated diminished DNA repair and lack of an acute DNA damage-induced cell cycle arrest, insight into the mechanistic re-wiring of DDR signaling during mitosis was only recently provided. Different mechanisms appear to be at play to inactivate specific signaling axes of the DDR network in mitosis. Importantly, mitotic cells not simply inactivate the entire DDR, but appear to mark their DNA damage for repair after mitotic exit.


## The dense arm

Same mechanics, different engine: BGE-small encodes the 24 917 passages and the claims
(with the testbed's query prefix), cosine scores on the full matrix, then the same MaxP,
the same distractor rule, the same sister filter. The dense scores are not
bit-reproducible across machines, so the run is encoded once, versioned, and read back.
The two engines agree on about a third of their distractors — two genuinely different
sources of hard documents, as in the paper's retriever check.

In [10]:
with open(method.BGE_RUN_FILE) as f:
    bge_run = json.load(f)
with open(method.BGE_BEST_FILE) as f:
    bge_best = json.load(f)
ndcg, recall = method.evaluate(bge_run)
print(f"bge on passages: nDCG@10 {ndcg:.3f}  Recall@100 {recall:.3f}")
print("bge distractors of claim 1140:", draws["1140"]["bge"][:3])
overlap = 0.0
for qid in claims:
    one = {method.split_id(pid)[0] for pid in draws[qid]["bm25"][:16]}
    two = {method.split_id(pid)[0] for pid in draws[qid]["bge"][:16]}
    overlap += len(one & two) / 16
print(f"abstracts shared by the first 16 bm25 and bge distractors: {100 * overlap / 188:.0f} % on average")

bge on passages: nDCG@10 0.723  Recall@100 0.963
bge distractors of claim 1140: ['4500832:6-10', '15648443:3-9', '308862:2-5']
abstracts shared by the first 16 bm25 and bge distractors: 32 % on average


## The gold window

The model reads a 100-word window of the gold too, never the raw abstract head: among
the windows of whole sentences totalling at most 100 words, take the one holding the
MOST complete evidence sets (a set counts from its first to its last sentence); ties go
to the most compact set, then the shortest window; the window then grows with its
neighbouring sentences, before then after, up to 100 words. A multi-gold claim picks the
gold with the best window. Two claims have no evidence set fitting 100 words anywhere
(1110 and 1241): their window widens to the most compact set, a documented exception.

In [11]:
doc, a, b, widened = method.gold_window(claim, sentences)
gold_pid = {"1140": f"{doc}:{a}-{b}"}
print("gold window of claim 1140:", gold_pid["1140"], " widened:", widened)
print()
print(method.passage_text(titles, sentences, doc, a, b))

gold window of claim 1140: 12009265:11-15  widened: False

Vitamins E and C in the prevention of prostate and total cancer in men: the Physicians' Health Study II randomized controlled trial. Neither vitamin E nor vitamin C had a significant effect on colorectal, lung, or other site-specific cancers. Adjustment for adherence and exclusion of the first 4 or 6 years of follow-up did not alter the results. Stratification by various cancer risk factors demonstrated no significant modification of the effect of vitamin E on prostate cancer risk or either agent on total cancer risk. CONCLUSIONS In this large, long-term trial of male physicians, neither vitamin E nor C supplementation reduced the risk of prostate or total cancer.


## One prompt, in full

Instruction from the design, its exact wording to be frozen at the pilot (no NO-RES, no
token cap, a citation request instead),
documents renumbered 1..n, the gold at the cell's position (far = first, mid = middle,
near = last), **raw prompt without any chat template** — the paper's own regime. Here
claim 1140 in the cell `bm25_k4_near`: 4 BM25 distractors, gold last.

In [12]:
top3 = {"1140": {"bm25": [], "bge": []}}
for d in list(run["1140"])[:3]:
    top3["1140"]["bm25"].append(best_pass["1140"][d])
for d in list(bge_run["1140"])[:3]:
    top3["1140"]["bge"].append(bge_best["1140"][d])
docs = method.cell_documents("bm25_k4_near", "1140", draws, gold_pid, top3)
prompt, mapping = method.build_prompt(claim["text"], docs, titles, sentences)
print(prompt)
print()
print("mapping:", mapping)

You are given scientific documents and a claim. Answer SUPPORT if the documents support the claim or CONTRADICT if they contradict it, then cite the document that justifies your answer, in the format: SUPPORT|CONTRADICT — Document [n].
Documents:
Document [1](Title: Regulation of E2Fs and senescence by PML nuclear bodies.) Blocking the functions of the Rb protein family or adding back E2Fs to PML-expressing cells can rescue their defects in E2F-dependent gene expression and cell proliferation, inhibiting the senescent phenotype. In benign prostatic hyperplasia, a neoplastic disease that displays features of senescence, PML was found to be up-regulated and forming nuclear bodies. In contrast, PML bodies were rarely visualized in prostate cancers. The newly defined PML/Rb/E2F pathway may help to distinguish benign tumors from cancers, and suggest E2F target genes as potential targets to induce senescence in human tumors.
Document [2](Title: Screening for Lung Cancer: U.S. Preventive Serv

## Token budgets (dry-run)

Every prompt of the grid — 46 cells on Phi-2, 73 on Llama-2 (k stops at 14: the dense
arm overflows the window at k=16, so the pre-declared rule removed that column), plus
the two mixed cells — is built and counted with the two real tokenizers before anything
runs on a GPU. Limits: 2048 and 4096 minus 30 tokens reserved for the answer. One
overflow would stop `method.py`; there is none, and `dry_run.txt` keeps the tightest
margin of each arm.

In [13]:
toks = method.load_tokenizers()
print("the k=4 prompt above:", len(toks["phi2"](prompt)["input_ids"]), "phi-2 tokens,",
      len(toks["llama2"](prompt)["input_ids"]), "llama-2 tokens")
print()
with open(method.DRY_FILE) as f:
    print(f.read())

/opt/homebrew/Caskroom/miniconda/base/envs/tmc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


the k=4 prompt above: 810 phi-2 tokens, 989 llama-2 tokens

phi2  (limit 2018 = 2048 - 30 ; 48 cells x 188 claims, no overflow; tightest margin per arm)
  closed  margin  1933  closed_book      claim 528    (85 tokens)
  gold    margin  1623  gold_only        claim 1241   (395 tokens)
  random  margin   202  random_k8_far    claim 275    (1816 tokens)
  bm25    margin   121  bm25_k8_far      claim 636    (1897 tokens)
  bge     margin    60  bge_k8_far       claim 338    (1958 tokens)
  top3    margin  1193  top3_bm25        claim 636    (825 tokens)
  mixed   margin   221  mixed_bm25       claim 811    (1797 tokens)
llama2  (limit 4066 = 4096 - 30 ; 75 cells x 188 claims, no overflow; tightest margin per arm)
  closed  margin  3961  closed_book      claim 528    (105 tokens)
  gold    margin  3580  gold_only        claim 1241   (486 tokens)
  random  margin   588  random_k14_far   claim 275    (3478 tokens)
  bm25    margin   482  bm25_k14_far     claim 636    (3584 tokens)
  bge     

## What ships

`bm25_run.json` + `bge_run.json` and their `*_best_passage.json` companions (300 claims
× 101 abstracts each, MaxP), `draws.json` (188 claims × 18 randoms + 18 BM25 + 18 BGE
distractors), `dry_run.txt` (tightest token margins). `method.py` never overwrites them:
once written, a recomputation must give back the same content or it stops. All control
numbers are asserted in `method.py:main()`: 188 claims (124 SUPPORT / 64 CONTRADICT, 10
multi-gold), 24 917 passages, BM25 nDCG@10 0.691 / Recall@100 0.921, distractor n°1 of
claim 1140 = `41710132:2-6`, gold windows maximal for 187/188 (1110 and 1241 widened
beyond 100 words), grid of 46 + 73 cells, zero token overflow.